# 00 – Data access: Kepler, TESS, JWST, Hubble
Run this first (CPU runtime is fine). It only *tests* each connection with tiny downloads. Nothing here trains a model.

In [ ]:
!pip -q install lightkurve astroquery pandas

## A. NASA Exoplanet Archive (labels + star/planet info) – TAP API

In [ ]:
import io, requests, pandas as pd
def archive(query):
    r = requests.get('https://exoplanetarchive.ipac.caltech.edu/TAP/sync', params={'query':query,'format':'csv'})
    return pd.read_csv(io.StringIO(r.text))
koi = archive("select top 5 kepid,koi_disposition,koi_period,koi_time0bk,koi_duration from cumulative")   # Kepler
toi = archive("select top 5 tid,tfopwg_disp,pl_orbper,pl_tranmid,pl_trandurh from toi")                   # TESS
print(koi); print(toi)

## B. Kepler + TESS light curves – `lightkurve` (talks to MAST for you)

In [ ]:
import lightkurve as lk
k = lk.search_lightcurve('Kepler-10', mission='Kepler', author='Kepler', cadence='long')[:1].download()
t = lk.search_lightcurve('TOI-700',   mission='TESS',   author='SPOC')[:1].download()
print(len(k.flux), 'Kepler points |', len(t.flux), 'TESS points')
k.plot(); t.plot();

## C. JWST + Hubble – `astroquery.mast` (finding observations)

In [ ]:
from astroquery.mast import Observations
jwst = Observations.query_criteria(obs_collection='JWST', target_name='K2-18')
hst  = Observations.query_criteria(obs_collection='HST', target_name='K2-18', instrument_name='WFC3/IR')
print(len(jwst), 'JWST obs |', len(hst), 'HST obs')
jwst['obs_id','instrument_name','target_name','t_exptime'][:5]

## D. Hubble UV of stars (MUSCLES) – for the stellar-XUV step

In [ ]:
muscles = Observations.query_criteria(obs_collection='HLSP', provenance_name='MUSCLES')
print(len(muscles), 'MUSCLES products'); muscles['target_name','obs_id'][:10]

### Important reality check
* **Kepler/TESS:** thousands of labelled stars → good for training.
* **JWST/Hubble:** only a few dozen planets have spectra, and raw files need a heavy reduction pipeline (Eureka!, jwst). So Model A trains on **simulated** data and is *applied* to the few real spectra you can reduce or find in published tables.